# 01 · Preprocess test001

按 run 逐个处理；连续数据不重复载入，trial 元数据保留，Laplacian 端点不混入主分支。

In [1]:
# [Setup] Dependencies and project configuration
from pathlib import Path
import sys, json
import numpy as np
import mne
ROOT = Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026')
PIPE = ROOT / 'color_cognition_pipeline' / 'analyse_0720'
sys.path.insert(0, str(PIPE))
import config
from utils.epochs import save_epochs, baseline_subtract, baseline_zscore
config.ensure_output_dirs()

In [2]:
# [Reference] Reuse Laplacian and exclude confirmed bad neighbors
from utils.preprocessing import contact_laplacian
def local_laplacian(raw, subject):
    return contact_laplacian(raw, raw.ch_names, config.BAD_CHANNELS.get(subject, ()))

In [3]:
# [ERP] One run at a time; save compressed epoch arrays and provenance
def process_erp_run(subject, task, set_path):
    raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
    raw.resample(config.SFREQ_ERP, verbose=False)
    raw.notch_filter(config.LINE_NOISE_HZ, verbose=False)
    raw.filter(1., 30., verbose=False)
    ref = local_laplacian(raw, subject)
    events, event_id = mne.events_from_annotations(ref, verbose=False)
    selected = {name: code for name, code in event_id.items() if name.startswith('Trigger-In:')}
    epochs = mne.Epochs(ref, events, event_id=selected, tmin=config.EPOCH_TMIN_S, tmax=config.EPOCH_TMAX_S, baseline=None, preload=True, reject_by_annotation=False, verbose=False)
    data = baseline_subtract(epochs.get_data(), epochs.times * 1000, tuple(v * 1000 for v in config.ERP_BASELINE_S)) * 1e6
    inverse = {code: name for name, code in selected.items()}
    trigger_names = [inverse[int(code)] for code in epochs.events[:, 2]]
    out = config.INTERMEDIATE_ROOT / subject / 'preprocessing' / f'task{task}_erp.npz'
    save_epochs(out, data, epochs.times * 1000, trigger_names, epochs.ch_names, {'task': task, 'reference': 'contact_laplacian', 'bad_channels': config.BAD_CHANNELS.get(subject, ()), 'baseline_ms': [-200, 0], 'units': 'microvolt', 'source': str(set_path)})
    del raw, ref, epochs, data

In [4]:
# [Run] Execute sequentially; HG extraction is kept as a separate cached stage
for task, run_name in config.RUNS.items():
    process_erp_run('test001', task, config.subject_raw_dir('test001') / f'{run_name}.set')
print('ERP preprocessing complete')

ERP preprocessing complete
